# Part 1 — Linear Methods for Multi-Omic Integration

This workshop notebook compares two simple linear strategies for integrating TCGA-BRCA multi-omics data:

1. **Early integration**: concatenate transcriptomics, proteomics, and methylation features, then fit one linear classifier.
2. **Late integration**: fit one linear classifier per omic, then aggregate their predictions.

The data are already processed, aligned, and complete. The patient ID is the index, and the supervised target is the `subtype` column.

## Learning objectives

By the end of this notebook you should be able to:

- Load already processed omics matrices.
- Use a single shared train/test split across all integration methods.
- Visualize how omics differ using PCA.
- Train single-omic linear baselines.
- Train and evaluate early integration by concatenation.
- Train and evaluate late integration by prediction averaging.
- Discuss why simple integration motivates patient-level multi-omic representations in the next part.

## Expected data layout

```text
../../data_tmp/TCGA-BRCA/
├── transcriptomics.pkl
├── proteomics.pkl
└── methylation.pkl
```

Each pickle should contain a pandas `DataFrame` with:

- rows = patients / samples
- index = patient ID
- columns = molecular features plus one metadata column named `subtype`

No imputation, sample alignment, missing-patient handling, or cross-validation is performed in this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 7
TEST_SIZE = 0.25
TARGET_COLUMN = "subtype"

DATA_DIR = Path("../../data_tmp/TCGA-BRCA")
OMICS_FILES = {
    "transcriptomics": DATA_DIR / "transcriptomics.pkl",
    "proteomics": DATA_DIR / "proteomics.pkl",
    "methylation": DATA_DIR / "methylation.pkl",
}

## 1. Load the already processed data

The only split we make here is between the molecular features and the `subtype` target. Patient IDs remain in the index.

In [ ]:
def load_omic_table(path: Path) -> pd.DataFrame:
    """Load one processed omic table from pickle."""
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file: {path}")
    df = pd.read_pickle(path)
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected {path.name} to contain a pandas DataFrame.")
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f"Expected a '{TARGET_COLUMN}' column in {path.name}.")
    return df

omics_tables = {name: load_omic_table(path) for name, path in OMICS_FILES.items()}

shape_summary = pd.DataFrame(
    {
        name: {
            "patients": df.shape[0],
            "total_columns": df.shape[1],
            "feature_columns": df.shape[1] - 1,
            "subtypes": df[TARGET_COLUMN].nunique(),
        }
        for name, df in omics_tables.items()
    }
).T

shape_summary

## 2. Create feature matrices and target vector

The samples are already aligned and complete. The assertions below are lightweight guardrails so the notebook fails loudly if the wrong files are loaded.

In [ ]:
# The first table defines the shared patient order and target.
reference_omic = next(iter(omics_tables))
reference_index = omics_tables[reference_omic].index

y = omics_tables[reference_omic][TARGET_COLUMN].astype(str)
X_omics = {}

for name, df in omics_tables.items():
    # Confirm that all omics use the same patients in the same order.
    assert df.index.equals(reference_index), f"{name} is not aligned to {reference_omic}."

    # Confirm that every omic carries the same target labels.
    assert df[TARGET_COLUMN].astype(str).equals(y), f"{name} has different subtype labels."

    # Keep all already processed molecular features.
    X_omics[name] = df.drop(columns=[TARGET_COLUMN])

print(f"Patients: {len(y)}")
display(y.value_counts().rename("n_patients"))
display(pd.DataFrame({name: X.shape for name, X in X_omics.items()}, index=["patients", "features"]).T)

## 3. Make one shared train/test split

The same patient split is reused for single-omic baselines, early integration, and late integration. This keeps all comparisons fair and easy to explain.

In [ ]:
patient_ids = y.index
train_ids, test_ids = train_test_split(
    patient_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

y_train = y.loc[train_ids]
y_test = y.loc[test_ids]

print(f"Training patients: {len(train_ids)}")
print(f"Test patients: {len(test_ids)}")

split_balance = pd.DataFrame({
    "train": y_train.value_counts(),
    "test": y_test.value_counts(),
}).fillna(0).astype(int)

split_balance

## 4. PCA view of each omic

PCA is not used for the classifiers below. It is only a quick visual check: if the panels separate patients differently, they may contain complementary information.

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = list(label_encoder.classes_)


def plot_pca_by_omic(X_omics, y_encoded, class_names):
    """Plot the first two PCs for each omic using all patients."""
    for name, X in X_omics.items():
        pca_pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=2, random_state=RANDOM_STATE)),
        ])
        scores = pca_pipe.fit_transform(X)
        evr = pca_pipe.named_steps["pca"].explained_variance_ratio_

        plt.figure(figsize=(5.5, 4.5))
        scatter = plt.scatter(scores[:, 0], scores[:, 1], c=y_encoded, s=25, alpha=0.8)
        plt.title(f"{name}: PCA")
        plt.xlabel(f"PC1 ({evr[0]:.1%} variance)")
        plt.ylabel(f"PC2 ({evr[1]:.1%} variance)")
        handles, _ = scatter.legend_elements()
        plt.legend(handles, class_names, title="subtype", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

plot_pca_by_omic(X_omics, y_encoded, class_names)

## 5. Model helpers

We use the same regularized linear classifier everywhere. Standardization is included inside the pipeline so scaling is learned only from the training data.

In [ ]:
def make_linear_classifier():
    """Regularized linear model for high-dimensional omics classification."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])


def evaluate_predictions(model_name, y_true, y_pred):
    """Return the metrics used throughout the tutorial."""
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def fit_predict(model, X_train, y_train, X_test):
    """Fit a model and return class predictions and probabilities for the test set."""
    fitted = clone(model).fit(X_train, y_train)
    pred = pd.Series(fitted.predict(X_test), index=X_test.index, name="prediction")
    proba = pd.DataFrame(fitted.predict_proba(X_test), index=X_test.index, columns=fitted.classes_)
    return fitted, pred, proba

## 6. Single-omic benchmarks

Before integrating data types, measure how far each omic gets on its own.

In [ ]:
results = []
single_omic_models = {}
single_omic_probabilities = {}

base_model = make_linear_classifier()

for name, X in X_omics.items():
    model, pred, proba = fit_predict(
        base_model,
        X.loc[train_ids],
        y_train,
        X.loc[test_ids],
    )
    single_omic_models[name] = model
    single_omic_probabilities[name] = proba
    results.append(evaluate_predictions(f"single omic: {name}", y_test, pred))

results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
results_df

## 7. Early integration by concatenation

Early integration joins all omic features into one wide matrix and trains one classifier.

This is easy and often strong, but it increases dimensionality and does not explicitly distinguish shared, redundant, and omic-specific signal.

In [ ]:
def prefix_columns(X: pd.DataFrame, omic_name: str) -> pd.DataFrame:
    """Add an omic prefix so features remain traceable after concatenation."""
    X_prefixed = X.copy()
    X_prefixed.columns = [f"{omic_name}::{col}" for col in X.columns]
    return X_prefixed

X_early = pd.concat(
    [prefix_columns(X, name) for name, X in X_omics.items()],
    axis=1,
)

print(f"Early integration matrix: {X_early.shape[0]} patients x {X_early.shape[1]} features")

early_model, early_pred, early_proba = fit_predict(
    base_model,
    X_early.loc[train_ids],
    y_train,
    X_early.loc[test_ids],
)

results.append(evaluate_predictions("early integration: concatenation", y_test, early_pred))
results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
results_df

## 8. Late integration by prediction averaging

Late integration trains one model per omic and averages the class probabilities. Each omic gets an equal vote by default.

In [ ]:
def late_integrated_probabilities(probability_tables, weights=None):
    """Average same-index, same-column probability tables from omic-specific models."""
    weights = weights or {name: 1.0 for name in probability_tables}
    weight_total = sum(weights.values())

    averaged = None
    for name, proba in probability_tables.items():
        weighted_proba = proba * weights[name]
        averaged = weighted_proba if averaged is None else averaged + weighted_proba

    return averaged / weight_total

late_proba = late_integrated_probabilities(single_omic_probabilities)
late_pred = late_proba.idxmax(axis=1).rename("prediction")

results.append(evaluate_predictions("late integration: equal weights", y_test, late_pred))
results_df = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=False)
results_df

## 9. Compare all linear methods

The main question is whether multi-omic integration improves over the best single-omic baseline.

In [ ]:
summary = pd.DataFrame(results).sort_values("balanced_accuracy", ascending=True)
display(summary.sort_values("balanced_accuracy", ascending=False))

plt.figure(figsize=(8, 4.5))
plt.barh(summary["model"], summary["balanced_accuracy"])
plt.xlabel("Balanced accuracy on held-out test set")
plt.title("Single-omic vs integrated linear models")
plt.tight_layout()
plt.show()

## 10. Inspect feature usage in the early-integration model

A linear model on concatenated features can be inspected through coefficient magnitudes. This helps demonstrate whether early integration spreads weight across omics or concentrates on a smaller set of features.

In [ ]:
def early_feature_importance(fitted_pipeline, feature_names):
    """Extract maximum absolute coefficient per feature."""
    coefficients = fitted_pipeline.named_steps["model"].coef_
    importance = np.abs(coefficients).max(axis=0)

    feature_importance = pd.DataFrame({
        "feature": feature_names,
        "abs_coefficient": importance,
    })
    feature_importance["omic"] = feature_importance["feature"].str.split("::", n=1).str[0]
    return feature_importance.sort_values("abs_coefficient", ascending=False)

feature_importance = early_feature_importance(early_model, X_early.columns)

TOP_N = min(100, len(feature_importance))
top_feature_usage = (
    feature_importance.head(TOP_N)["omic"]
    .value_counts()
    .rename_axis("omic")
    .reset_index(name=f"features_in_top_{TOP_N}")
)

display(top_feature_usage)
display(feature_importance.head(20))

plt.figure(figsize=(6, 4))
plt.bar(top_feature_usage["omic"], top_feature_usage[f"features_in_top_{TOP_N}"])
plt.ylabel(f"Count among top {TOP_N} coefficients")
plt.title("Which omics dominate early-integration coefficients?")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 11. Cross-omic redundancy among highly weighted features

Concatenation does not explicitly model correlation between features. Here we look at correlations among highly weighted features from different omics.

In [ ]:
def top_features_per_omic(feature_importance, n_per_omic=20):
    """Return the top coefficient features within each omic."""
    selected = []
    for _, group in feature_importance.groupby("omic"):
        selected.extend(group.head(n_per_omic)["feature"].tolist())
    return selected

selected_features = top_features_per_omic(feature_importance, n_per_omic=20)
X_selected = X_early.loc[:, selected_features]

corr = X_selected.corr().abs()
rows = []

for i, feature_1 in enumerate(corr.columns):
    omic_1 = feature_1.split("::", 1)[0]
    for feature_2 in corr.columns[i + 1:]:
        omic_2 = feature_2.split("::", 1)[0]
        if omic_1 != omic_2:
            rows.append({
                "feature_1": feature_1,
                "feature_2": feature_2,
                "omic_pair": f"{omic_1} ↔ {omic_2}",
                "abs_correlation": corr.loc[feature_1, feature_2],
            })

cross_omic_corr = pd.DataFrame(rows).sort_values("abs_correlation", ascending=False)
display(cross_omic_corr.head(20))

plt.figure(figsize=(6, 4))
plt.hist(cross_omic_corr["abs_correlation"], bins=30)
plt.xlabel("Absolute correlation")
plt.ylabel("Feature-pair count")
plt.title("Cross-omic correlation among highly weighted features")
plt.tight_layout()
plt.show()

## 12. Late-integration weight sensitivity

Equal weighting is simple, but it encodes a strong assumption: each omic should contribute equally. This small test boosts one omic at a time and checks whether the held-out accuracy changes.

In [ ]:
weight_rows = []
base_weights = {name: 1.0 for name in X_omics}

# Equal weights.
proba = late_integrated_probabilities(single_omic_probabilities, weights=base_weights)
pred = proba.idxmax(axis=1)
weight_rows.append({
    "weighting": "equal weights",
    **evaluate_predictions("late", y_test, pred),
})

# Boost each omic while keeping the same trained single-omic models and same test split.
for boosted_omic in X_omics:
    weights = base_weights.copy()
    weights[boosted_omic] = 2.0
    proba = late_integrated_probabilities(single_omic_probabilities, weights=weights)
    pred = proba.idxmax(axis=1)
    weight_rows.append({
        "weighting": f"boost {boosted_omic}",
        **evaluate_predictions("late", y_test, pred),
    })

weight_sensitivity = (
    pd.DataFrame(weight_rows)
    .drop(columns="model")
    .sort_values("balanced_accuracy", ascending=False)
)

display(weight_sensitivity)

plt.figure(figsize=(7, 4))
plt.barh(weight_sensitivity["weighting"], weight_sensitivity["balanced_accuracy"])
plt.xlabel("Balanced accuracy on held-out test set")
plt.title("Late-integration weighting sensitivity")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 13. Confusion matrix and classification report

Use this section to discuss which subtypes are easier or harder for the best simple model.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
print(f"Best model by balanced accuracy: {best_model_name}")

if best_model_name == "early integration: concatenation":
    best_pred = early_pred
elif best_model_name == "late integration: equal weights":
    best_pred = late_pred
else:
    best_omic = best_model_name.split(": ", 1)[1]
    best_pred = single_omic_models[best_omic].predict(X_omics[best_omic].loc[test_ids])
    best_pred = pd.Series(best_pred, index=test_ids)

cm = pd.DataFrame(
    confusion_matrix(y_test, best_pred, labels=class_names),
    index=[f"true: {c}" for c in class_names],
    columns=[f"pred: {c}" for c in class_names],
)

display(cm)
print(classification_report(y_test, best_pred))

## Takeaways

1. **Different omics capture different sources of variation.** PCA gives a visual entry point for comparing transcriptomics, proteomics, and methylation.
2. **Single-omic models are essential baselines.** Integration should be judged against the best individual data type, not just against intuition.
3. **Early integration is simple but blunt.** Concatenation can improve accuracy, but it expands the feature space and can concentrate coefficients in a subset of omics or features.
4. **Highly weighted features can still be redundant.** Correlated cross-omic features show why concatenation does not automatically learn structured multi-omic relationships.
5. **Late integration is interpretable but weight-sensitive.** Equal voting is clear, but changing omic weights can change predictions and performance.

## Bridge to the next part

These linear baselines point toward the next workshop objective: instead of representing a patient as three separate matrices or one very wide concatenated vector, we want to move toward an **individual multi-omic profile per patient**. The next part can ask how to learn representations that preserve patient-level information while modelling shared and omic-specific structure more directly.